# Stage 2 - Instruction Fine-Tuning (SFT)

**Goal:** teach the model to answer questions - map a natural-language question to either
the correct **table name** (schema discovery) or a valid **SQL query** (support queries).

Data: `ecomm-db-instruction` on the Hugging Face Hub (215 `{instruction, response}` examples).

In [ ]:
# Install Unsloth (needs a CUDA GPU). If Colab prompts to restart after install, do it.
%%capture
!pip install --upgrade unsloth unsloth_zoo
# Ensure a MODERN TRL (provides SFTConfig/DPOConfig + processing_class) that matches new transformers.
!pip install --upgrade "trl>=0.13.1"

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - set Runtime > Change runtime type > T4 GPU")

## 1. Config

In [ ]:
# --- Datasets live on the Hugging Face Hub (pushed via scripts/push_to_hf.py) ---
HF_USER = "Rajesh507"   # <-- your Hugging Face username
DS_NONINSTRUCT = f"{HF_USER}/ecomm-db-noninstruct"
DS_INSTRUCTION = f"{HF_USER}/ecomm-db-instruction"
DS_PREFERENCE  = f"{HF_USER}/ecomm-db-preference"

# If the datasets are PRIVATE, log in first (needs a read token):
# from huggingface_hub import login; login()

In [ ]:
# Persist trained adapters to the Hugging Face Hub (no Google Drive needed).
from huggingface_hub import login
login()   # paste a WRITE token: https://huggingface.co/settings/tokens

ADAPTER_STAGE1 = f"{HF_USER}/ecomm-db-stage1-noninstruct"
ADAPTER_STAGE2 = f"{HF_USER}/ecomm-db-stage2-sft"
ADAPTER_STAGE3 = f"{HF_USER}/ecomm-db-stage3-dpo"
print("Adapters will be pushed under:", HF_USER)

## 2. Load model
Start from the base model, or continue from the Stage-1 adapter for a domain-adapted start.

In [ ]:
MODEL_NAME = "unsloth/Qwen2.5-Coder-1.5B"   # Coder variant is stronger at SQL. Alt: "unsloth/Qwen2.5-1.5B"
MAX_SEQ_LEN = 2048

In [ ]:
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_NAME,   # or ADAPTER_STAGE1 to continue from the Stage-1 adapter
    max_seq_length = MAX_SEQ_LEN,
    dtype = None,
    load_in_4bit = True,
)

In [ ]:
from unsloth import FastLanguageModel
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,                       # LoRA rank
    lora_alpha = 16,              # scaling
    lora_dropout = 0,             # 0 is optimized in Unsloth
    bias = "none",
    target_modules = ["q_proj","k_proj","v_proj","o_proj",
                      "gate_proj","up_proj","down_proj"],
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

## 3. Load and format the instruction dataset

In [ ]:
# A single, consistent prompt template used across ALL three stages.
PROMPT = """Below is a question about the client e-commerce database schema. \
Write a response that correctly answers it, giving the exact table name(s) or a valid SQL query.

### Question:
{}

### Answer:
{}"""

In [ ]:
from datasets import load_dataset
EOS = tokenizer.eos_token

def format_examples(batch):
    return {"text": [PROMPT.format(i, r) + EOS
                     for i, r in zip(batch["instruction"], batch["response"])]}

ds = load_dataset(DS_INSTRUCTION, split="train").map(format_examples, batched=True)
print(ds)
print(ds[0]["text"])

## 4. Train (SFT)

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    processing_class = tokenizer,   # new TRL API (was 'tokenizer=')
    train_dataset = ds,
    args = SFTConfig(
        dataset_text_field = "text",
        max_seq_length = MAX_SEQ_LEN,
        dataset_num_proc = 2,
        packing = False,
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs_stage2",
        report_to = "none",
    ),
)
trainer_stats = trainer.train()

## 5. Push the SFT adapter to the Hugging Face Hub

In [ ]:
model.push_to_hub(ADAPTER_STAGE2, token=True)
tokenizer.push_to_hub(ADAPTER_STAGE2, token=True)
print("Pushed SFT adapter to:", ADAPTER_STAGE2)

## 6. Inference after SFT

In [ ]:
FastLanguageModel.for_inference(model)

def ask(question, max_new_tokens=128):
    text = PROMPT.format(question, "")
    inputs = tokenizer(text, return_tensors="pt").to("cuda")
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, temperature=0.2, do_sample=False)
    return tokenizer.decode(out[0], skip_special_tokens=True).split("### Answer:")[-1].strip()

for q in [
    "Which table stores product images?",
    "Give me a query to find unique orders for a customer.",
    "Find all shipments that have not been delivered.",
]:
    print("Q:", q); print("A:", ask(q)); print("-"*60)

### Next
Proceed to `dpo_alignment.ipynb`.